In [2]:
# Get F1-scores for each of the target classes, 
# determined on predictions from TEST sets, 
# and used as weights when getting the pollen counts with the fractional approach

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))
from config1 import CLASSIFICATION_DATA, classes54


In [11]:
import pickle
from sklearn.metrics import f1_score
import pandas as pd
from collections import Counter

In [23]:
p_model_outputs = CLASSIFICATION_DATA / "4.MODEL_PREDICTIONS"



In [ ]:

classes17 =  ['Buxus', 'Cupressaceae', 'Fraxinus',  'IndetBlurry', 'IndetCovered', 'Lycopodium', 'NonPollen', 'Olea', 
               'Phillyrea', 'Pinaceae', 'Pistacia', 'Plantago', 'Poaceae', 'QuercusDeciduous', 'QuercusIlex', 'VitisF', 'VitisS']


classes18 = classes17 + ["Other"]
classes18 = sorted(classes18)




In [25]:

def get_f1s_modelweights(mp, cval_id): 
    """ 
    f1-scores are determined per model j and per class i (out of 18 classes)
    Only the images "env" are considered for the evaluation
    Images from non-target taxa are grouped in the class "other"
    Computed f1-scores per model j and per class i will be directly used as weight_ij
    """
    p_pred = mp + f"{cval_id}_pred_classes.pkl"
    p_true = mp + f"{cval_id}_true_classes.pkl"
    p_filename = mp + f"{cval_id}_images.pkl"
    
    
    with open(p_true, 'rb') as f:
        true = pickle.load(f)
    
    with open(p_pred, 'rb') as f:
        pred = pickle.load(f)
    
    with open(p_filename, 'rb') as f:
        filename = pickle.load(f)

    true = [classes54[el] for el in true]
    pred = [classes54[el] for el in pred]
    
    true = ["Fraxinus" if "Fraxinus" in el else el for el in true]
    pred = ["Fraxinus" if "Fraxinus" in el else el for el in pred]
    
    true_env = [el for e, el in enumerate(true) if filename[e].split('/')[-1][:3]=="env"]
    pred_env = [el for e, el in enumerate(pred) if filename[e].split('/')[-1][:3]=="env"]
    true_env_grouped =  [el if el in classes17 else "Other"  for el in true_env] 
    pred_env_grouped =  [el if el in classes17 else "Other"  for el in pred_env] 
    
    f1_per_class = f1_score(true_env_grouped, pred_env_grouped, labels=classes18,  average=None)
    dict_f1score = {cls: float(score) for cls, score in zip(classes18, f1_per_class)}
    counter_per_class = dict(Counter(true_env_grouped))
    # print("counter sorted per effectif", dict(sorted(counter_per_class.items(), key=lambda x: x[1])))

    return counter_per_class, dict_f1score

In [ ]:


li_csv=[]
global_weights={}
for ids in [0,1,2,4,3,5,6,7,8,9]:
    cval_id = f"fold{ids+1}"
    print(cval_id)
    counter_per_class, dict_f1score = get_f1s_modelweights(str(p_model_outputs)+"/", cval_id)
    global_weights[cval_id] = dict_f1score
    for classk in dict_f1score.keys():
        li_csv+=[[cval_id, classk, dict_f1score[classk], counter_per_class[classk]]]
df = pd.DataFrame(li_csv, columns=["cval", "classe", 'f1score', "counter"])

# df.to_csv( CLASSIFICATION_DATA /"globalweights_f1scores_18cl.csv", index=False)
df


fold1
fold2
fold3
fold5
fold4
fold6
fold7
fold8
fold9
fold10


,cval,classe,f1score,counter
0,fold1,Buxus,0.988235,84
1,fold1,Cupressaceae,0.956989,91
2,fold1,Fraxinus,0.877193,28
3,fold1,IndetBlurry,0.953488,44
4,fold1,IndetCovered,0.893617,49
...,...,...,...,...
175,fold10,Poaceae,0.989583,96
176,fold10,QuercusDeciduous,0.720000,13
177,fold10,QuercusIlex,0.941176,68
178,fold10,VitisF,0.966102,57
